In [34]:
import pandas as pd

In [35]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [36]:
from src.data.db_connection import load_movies

df = load_movies()
print("Загружены данные из БД, количество строк: ", len(df))
print(df.head())
print(df.columns.tolist())

c:\Users\User\Desktop\Учеба\python\movie-tagging-service\src\data\db_connection.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Загружены данные из БД, количество строк:  797
   id         movie_name  year  rating     runtime  meta_score  age_rating  \
0   1                1+1  2011     8.8   112 минут          57        18.0   
1   2       Интерстеллар  2014     8.7   169 минут          74        18.0   
2   3  Побег из Шоушенка  1994     9.1  142 минуты          82        16.0   
3   4   Остров проклятых  2009     8.5   138 минут          63        18.0   
4   5        Джентльмены  2019     8.6   113 минут          51        18.0   

                                            overview  \
0  Аристократ на коляске нанимает в сиделки бывше...   
1  Когда засуха, пыльные бури и вымирание растени...   
2  Бухгалтер Энди Дюфрейн обвинён в убийстве собс...   
3  Два американских судебных пристава отправляютс...   
4  Один ушлый американец ещё со студенческих лет ...   

                           genres                    directors  \
0                  Драма, Комедия  Оливье Накаш, Эрик Толедано   
1  Драма, Прикл

Проверка пропущенных значений

In [37]:
df.isnull().sum()

id             0
movie_name     0
year           0
rating         0
runtime        0
meta_score     0
age_rating    44
overview       1
genres         0
directors      0
countries      1
dtype: int64

Заполним вручную пропуски

In [38]:
df[["movie_name", "year", "age_rating"]][df["age_rating"].isnull()]

,movie_name,year,age_rating
368,Банши Инишерина,2022,NaN
372,Достать ножи: Стеклянная луковица,2022,NaN
387,Вавилон,2022,NaN
388,Нэчжа,2019,NaN
391,Меню,2022,NaN
401,El Camino: Во все тяжкие,2019,NaN
414,История о супружестве,2019,NaN
447,Ауру,2025,NaN
451,Семья в аренду,2025,NaN
458,Грация,2025,NaN


In [39]:
def fill_column(row, column):
    if pd.notna(row[column]):
        return row[column]
    
    if row['movie_name'] in explicit_age_ratings:
        return explicit_age_ratings[row['movie_name']]
    
    return None

In [40]:
explicit_age_ratings = {
    'Банши Инишерина': 16.0,
    'Достать ножи: Стеклянная луковица': 12.0,
    'Вавилон': 16.0,
    'Меню': 16.0,
    'El Camino: Во все тяжкие': 18.0,
    'История о супружестве': 16.0,
    'Оппенгеймер': 18.0,
    'Оставь мир позади': 16.0,
    'Оставленные': 12.0,
    'Форсаж 10': 12.0,
    'Кот в сапогах 2: Последнее желание': 6.0,
    'Подземелья и драконы: Честь среди воров': 12.0,
    'Аватар: Путь воды': 12.0,
    'Холодное сердце': 6.0,
    'Ирландец': 18.0,
    'Злая: Сказка о ведьме Запада': 6.0,
    'На Западном фронте без перемен': 18.0,
    'ВАЛЛ·И': 0.0,
    'Три кота. Зимние каникулы': 0.0,
    'Лунтик. Возвращение домой': 0.0,
    'Магическая битва 0. Фильм': 16.0,
    'Одинокие волки': 16.0,
    'Ловушка': 16.0,
    'Моя вина': 16.0,
    'Не говори никому': 16.0,
    'Клуб убийств по четвергам': 12.0,
    'Источник вечной молодости': 12.0,
    'Джей Келли': 18.0,
    'Грация': 12.0,
    'Ауру': 12.0,
    'Семья в аренду': 12.0,
    'Я ругаюсь': 18.0,
    'Даже если эта любовь исчезнет сегодня ночью': 16.0,
    'Последняя капля': 16.0,
    'Баллада о маленьком игроке': 16.0,
    'Школьный автобус': 16.0,
    'Хамнет: История, вдохновившая «Гамлета»': 12.0,
    'Главы государств': 16.0,
    'Отпуск на двоих': 12.0,
    'Всемирный потоп': 16.0,
    'Ещё одна простая просьба': 16.0,
    'Грязная игра': 16.0,
    'Подай знак': 16.0,
    'Нэчжа': 6.0,
}

df['age_rating'] = df.apply(fill_column, column='age_rating', axis=1)

print(f"Пропусков в age_rating после заполнения: {df['age_rating'].isna().sum()}")

Пропусков в age_rating после заполнения: 0


In [41]:
df[["movie_name", "year", "overview"]][df["overview"].isnull()]

,movie_name,year,overview
181,Американская кабала,2025,NaN


In [42]:
df.loc[df['movie_name'] == 'Американская кабала', 'overview'] = ("Модератор социальной сети Дейзи Мориарти ответственна за удаление оскорбительного и опасного контента. Однажды Дейзи видит настолько шокирующее видео, что она падает в обморок. Начальство и представители властей уверены, что видео является постановочным, однако Дейзи считает, что на записи было запечатлено настоящее преступление. Девушка решает самостоятельно вычислить виновного и привлечь его к правосудию.")

In [43]:
df[["movie_name", "year", "countries"]][df["countries"].isnull()]

,movie_name,year,countries
143,Хищник: Убийца убийц,2025,NaN


In [44]:
df.loc[df['movie_name'] == 'Хищник: Убийца убийц', 'countries'] = 'США'

In [45]:
df.isnull().sum()

id            0
movie_name    0
year          0
rating        0
runtime       0
meta_score    0
age_rating    0
overview      0
genres        0
directors     0
countries     0
dtype: int64

In [46]:
data_movies = df.to_csv("../data/processed/data_movies.csv", index=False)